# 01 — Data Extraction & Column Inventory

**NST DVA Capstone 2 · NASA Planetary Defense**  
**Team:** SectionC_G-09  
**Dataset:** NASA/JPL Asteroid Close Approaches (2015–2035) + Near-Earth Asteroid Catalogue (2025)

---

## Objectives

- Load both raw datasets from `data/raw/`
- Inspect shapes, dtypes, and sample rows
- Document all original abbreviated column names
- Preview the rename maps that will be applied in notebook 02

> ⚠️ **No modifications are made to the raw files in this notebook.** This is a read-only audit step.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# ── Path setup ──────────────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent  # notebooks/ → project root
RAW_DIR      = PROJECT_ROOT / 'data' / 'raw'
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data dir : {RAW_DIR}')

## 1.1 — Load Raw Datasets

In [ ]:
# Load Close Approaches (2015–2035)
close_path = RAW_DIR / 'asteroid_close_approaches_2015_2035.csv'
close_raw  = pd.read_csv(close_path)

# Load NEA Catalogue (2025 snapshot)
nea_path = RAW_DIR / 'near_earth_asteroids_2025.csv'
nea_raw  = pd.read_csv(nea_path, low_memory=False)

print('=== Close Approaches (Raw) ===')
print(f'  Shape : {close_raw.shape}')

print('\n=== Near-Earth Asteroids (Raw) ===')
print(f'  Shape : {nea_raw.shape}')

## 1.2 — Close Approaches: Column Audit

In [ ]:
print('Close Approaches — Column Overview')
print('=' * 55)
for col in close_raw.columns:
    null_pct = close_raw[col].isna().mean() * 100
    print(f'  {col:<35} {str(close_raw[col].dtype):<10} nulls: {null_pct:.1f}%')

In [ ]:
close_raw.head(5)

## 1.3 — NEA Catalogue: Column Audit

> **Note:** This dataset uses JPL SBDB abbreviated names for orbital elements.
> Fields like `H`, `e`, `a`, `i`, `q`, `ad`, `per`, `n` are standard orbital mechanics notation
> but are not self-explanatory for general analysis. These will be renamed in notebook 02.

In [ ]:
print('NEA Catalogue — Column Overview (⚠️ abbreviated names)')
print('=' * 65)
for col in nea_raw.columns:
    null_pct = nea_raw[col].isna().mean() * 100
    print(f'  {col:<35} {str(nea_raw[col].dtype):<10} nulls: {null_pct:.1f}%')

In [ ]:
nea_raw.head(5)

## 1.4 — Rename Maps Preview

The following rename maps will be applied in notebook 02 to give all columns fully descriptive names.

### NEA Dataset — Orbital Element Renames

In [ ]:
from etl_pipeline import NEA_RENAME_MAP, CLOSE_RENAME_MAP

print('NEA Column Renames')
print(f"{'Original':<30} → {'New Descriptive Name':<40}")
print('-' * 72)
for orig, new in NEA_RENAME_MAP.items():
    print(f'  {orig:<28} → {new}')

In [ ]:
print('Close Approaches Column Renames')
print(f"{'Original':<30} → {'New Descriptive Name':<40}")
print('-' * 72)
for orig, new in CLOSE_RENAME_MAP.items():
    print(f'  {orig:<28} → {new}')

## 1.5 — Data Quality Snapshot

In [ ]:
print('=== Close Approaches — Missing Value Summary ===')
missing_close = close_raw.isnull().sum()
missing_close = missing_close[missing_close > 0]
if len(missing_close) == 0:
    print('  No missing values found.')
else:
    print(missing_close.to_string())

print('\n=== NEA Catalogue — Missing Value Summary ===')
missing_nea = nea_raw.isnull().sum()
missing_nea = missing_nea[missing_nea > 0]
if len(missing_nea) == 0:
    print('  No missing values found.')
else:
    print(missing_nea.to_string())

In [ ]:
print('=== Duplicate Row Check ===')
print(f'  Close Approaches duplicates : {close_raw.duplicated().sum():,}')
print(f'  NEA Catalogue duplicates    : {nea_raw.duplicated().sum():,}')

## 1.6 — Extraction Summary

| Dataset | Rows | Columns | Key Issues |
|---|---|---|---|
| Close Approaches (raw) | ~95k+ | 13 | Some nulls in velocity_infinity_km_s, absolute_magnitude |
| NEA Catalogue (raw) | ~36k+ | 29 | Abbreviated column names; nulls in diameter, albedo, rot_per |

**Next step → Notebook 02: Cleaning & Column Renaming**